In [2]:
import polars as pl
from datetime import date
# Import requested libraries

In [3]:
df = pl.read_csv("../data/processed/online_retail_clean.csv", try_parse_dates = True)
# Read the dataset

In [4]:
df = df.with_columns(
    (pl.lit(date(2010, 12, 10)) - df["invoice_date"]).alias("recency")
)
# Calculate the recency of each customer's last purchase

In [5]:
frequency = df.group_by("customer_id").agg(
    pl.col("invoice").n_unique().alias("frequency")
)
# Calculate the number of unique invoices for each customer

In [6]:
df = df.with_columns(
    (pl.col("quantity") * pl.col("price")).alias("monetary")
)
# Calculate the monetary value of each transaction

In [7]:
customer_monetary = df.group_by("customer_id").agg(
    pl.col("monetary").sum().alias("total_monetary")
)
# Calculate the total monetary value for each customer

In [20]:
# Keep only positive-quantity transactions for RFM analysis.
df_rfm = df.filter(pl.col("quantity") > 0)

rfm = df_rfm.group_by("customer_id").agg(
    (date(2010, 12, 10) - pl.col("invoice_date").max()).dt.total_days().alias("recency"),
    pl.col("invoice").n_unique().alias("frequency"),
    pl.col("monetary").sum().alias("total_monetary")
)
rfm
# Calculate RFM metrics for each customer

customer_id,recency,frequency,total_monetary
i64,i64,u32,f64
15774,54,5,1017.54
15339,57,5,1246.95
17257,72,2,189.78
13303,36,1,61.72
14044,54,6,2431.28
…,…,…,…
16693,47,3,701.62
17810,304,2,373.16
14484,37,3,749.06
